#### **Question 2 — House Price Prediction**
Goal: Predict house prices, `SalePrice`, using regression models.

**Key corrections in this version:**
- `NaN` values that the data dictionary defines as *"feature not present"* are recoded to `'None'` (a new category) — they are **not** dropped.
- `MSSubClass` is recast from integer to string (it is a building-type code, not an ordered number).
- `Utilities` is dropped — near-zero variance (99.9% identical).
- `LotFrontage` (genuinely missing) is imputed via neighbourhood median.
- `ElasticNet` and `XGBoost` are added to the model comparison.
- Alpha tuning for Ridge, Lasso, and ElasticNet (Lab 7 style).
- Full test-set evaluation: RMSE, MAE, R².

---
## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model   import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble       import RandomForestRegressor
from sklearn.pipeline       import Pipeline
from sklearn.compose        import ColumnTransformer
from sklearn.impute         import SimpleImputer
from sklearn.preprocessing  import OneHotEncoder, StandardScaler
from sklearn.metrics        import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

---
## Step 2 — Load & Inspect the Data

The dataset has **2 908 rows** and **79 predictor columns** plus `SalePrice`.  
The index runs 1–2 919 with 11 gaps (a Kaggle train/test split artefact).

In [ ]:
data = pd.read_csv('house_prices.csv', index_col='Id')

print("Shape:", data.shape)
display(data.head())

# The 11 missing index values
missing_indices = set(range(1, 2920)) - set(data.index)
print("Missing index values:", sorted(missing_indices))

print("\nData types:")
print(data.dtypes.value_counts())

---
## Step 3 — Exploratory Data Analysis

### 3a — Target distribution and log-transform

`SalePrice` is right-skewed (skewness ≈ 1.57).  
Applying `log1p` gives skewness ≈ 0.04 — much closer to the symmetric distribution  
that linear models assume for their residuals.

In [ ]:
print(data['SalePrice'].describe().round(0))
print("\nSkewness (raw):", data['SalePrice'].skew().round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(data['SalePrice'], bins=40, edgecolor='black')
axes[0].set_title("SalePrice — raw")
axes[0].set_xlabel("SalePrice")

log_price = np.log1p(data['SalePrice'])
axes[1].hist(log_price, bins=40, edgecolor='black')
axes[1].set_title("log(SalePrice + 1)")
axes[1].set_xlabel("log(SalePrice + 1)")

print("Skewness (log):", log_price.skew().round(4))
plt.tight_layout()
plt.show()

### 3b — Correlations with SalePrice

The strongest numeric predictors are overall quality (`OverallQual`), above-ground living  
area (`GrLivArea`), total basement area (`TotalBsmtSF`), and garage capacity (`GarageCars`).

In [ ]:
corr = data.corr(numeric_only=True)['SalePrice'].sort_values(ascending=False)
print("Top positive correlations:")
print(corr.head(15))
print("\nTop negative correlations:")
print(corr.tail(8))

# Scatter plots for four most correlated features
top_features = ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'GarageCars']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, top_features):
    ax.scatter(data[col], data['SalePrice'], alpha=0.3, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel('SalePrice')
    ax.set_title(f'vs {col}')
plt.suptitle("SalePrice vs Top Numeric Predictors")
plt.tight_layout()
plt.show()

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 3c — Skewness of numeric predictors

Many area/count columns are zero for most houses, producing extreme right-skew.  
`StandardScaler` centres and scales these, but does not remove skewness.  
A further improvement (beyond the scope here) would be to apply `log1p` to  
predictors with |skew| > 1 as well.

In [ ]:
num_feats = data.select_dtypes(include=np.number).drop(columns='SalePrice').columns
skews = data[num_feats].skew().sort_values(ascending=False)
print("Numeric predictors with |skew| > 1:")
print(skews[abs(skews) > 1])

### 3d — Understanding the missing values  *(the critical EDA step)*

In this dataset **`NaN` does not always mean "data is unknown"**.  
The Ames Housing data dictionary documents many columns where `NaN` means  
*"the house does not have that feature"*.  Dropping or blindly imputing these  
columns would discard real, informative signal.

We classify every column with missing values into one of four groups:

| Group | Columns | Meaning of NaN | Correct action |
|---|---|---|---|
| **A — no feature (categorical)** | PoolQC, MiscFeature, Alley, Fence, FireplaceQu, GarageType/Finish/Qual/Cond, BsmtQual/Cond/Exposure/FinType1/FinType2, MasVnrType | Feature absent | Fill with `'None'` |
| **B — no feature (numeric)** | MasVnrArea, GarageYrBlt, BsmtFullBath, BsmtHalfBath | No garage/basement → count/year = 0 | Fill with `0` |
| **C — genuinely unknown** | LotFrontage | Recording gap | Impute with neighbourhood median |
| **D — near-constant** | Utilities | 99.9 % identical (`AllPub`) | Drop — no predictive power |
| **E — single-row true NaN** | MSZoning (3), Exterior1st/2nd (1), KitchenQual (1), Electrical (1), SaleType (1) | Rare recording error | Fill with mode |

In [ ]:
# ── Evidence for Group A ────────────────────────────────────────────────────

# FireplaceQu: NaN appears ONLY when Fireplaces == 0
fp_no   = ((data['Fireplaces'] == 0) & data['FireplaceQu'].isna()).sum()
fp_yes  = ((data['Fireplaces'] > 0) & data['FireplaceQu'].isna()).sum()
print(f"FireplaceQu NaN where Fireplaces=0 : {fp_no:4d}  (all NaN rows)")
print(f"FireplaceQu NaN where Fireplaces>0 : {fp_yes:4d}  (expected 0)")

# Garage: all 5 garage columns share (nearly) the same NaN rows
gcols  = ['GarageType','GarageFinish','GarageQual','GarageCond','GarageYrBlt']
g_null = data[gcols].isna().sum(axis=1)
print(f"\nGarage — rows with ALL 5 cols NaN (no garage) : {(g_null == 5).sum()}")
print(f"Garage — rows with ANY col NaN                  : {(g_null > 0).sum()}")

# Basement: same pattern
bcols  = ['BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2']
b_null = data[bcols].isna().sum(axis=1)
print(f"\nBasement — rows with ALL 5 cols NaN (no basement): {(b_null == 5).sum()}")
print(f"Basement — rows with ANY col NaN                   : {(b_null > 0).sum()}")

# PoolQC: 3 rows where PoolArea > 0 but PoolQC is NaN → genuinely missing
pool_exc = data[data['PoolQC'].isna() & (data['PoolArea'] > 0)]
print(f"\nPoolQC NaN but PoolArea > 0 (genuinely missing): {len(pool_exc)} rows")

# ── Evidence for Group D ─────────────────────────────────────────────────────
print(f"\nUtilities value counts:")
print(data['Utilities'].value_counts(dropna=False))
print("→ 2905/2908 = AllPub.  Zero variance ≈ useless predictor.")

# ── MSSubClass — integer codes that are categories ───────────────────────────
print(f"\nMSSubClass unique values: {sorted(data['MSSubClass'].unique())}")
print("→ These are building-type codes (20=1-story, 60=2-story, etc.) — not ordered numbers.")

In [ ]:
miss = data.isnull().sum()
miss_pct = (miss / len(data) * 100).round(1)
miss_df = pd.DataFrame({'NaN count': miss, 'NaN %': miss_pct})
miss_df = miss_df[miss_df['NaN count'] > 0].sort_values('NaN count', ascending=False)
display(miss_df)

---
## Step 4 — Targeted Data Cleaning

Each fix is motivated directly by the evidence gathered in Step 3d.

In [ ]:
df = data.copy()

# ── Group A: semantic NaN → 'None' category ─────────────────────────────────
# NaN means the house has no pool / fireplace / garage / basement / veneer.
# We encode this as an explicit category 'None' so the model can learn from it.
none_fill_cols = [
    'PoolQC',                                        # no pool
    'MiscFeature',                                   # no misc feature
    'Alley',                                         # no alley access
    'Fence',                                         # no fence
    'FireplaceQu',                                   # no fireplace (proven by Fireplaces==0)
    'GarageType', 'GarageFinish',                   # no garage
    'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure',         # no basement
    'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType',                                    # no masonry veneer
]
for col in none_fill_cols:
    df[col] = df[col].fillna('None')

# ── Group B: semantic NaN → 0 ───────────────────────────────────────────────
# No garage → year built and bath counts are meaningless; set to 0.
df['GarageYrBlt']  = df['GarageYrBlt'].fillna(0)
df['MasVnrArea']   = df['MasVnrArea'].fillna(0)
df['BsmtFullBath'] = df['BsmtFullBath'].fillna(0)   # 1 row: no basement
df['BsmtHalfBath'] = df['BsmtHalfBath'].fillna(0)

# ── Group C: LotFrontage — impute via neighbourhood median ─────────────────
# Up to 55 % missing in some neighbourhoods; the neighbourhood median is the
# best proxy because street frontages cluster by locality.
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'] \
                      .transform(lambda x: x.fillna(x.median()))

# ── Group D: Utilities — drop (near-zero variance) ──────────────────────────
df = df.drop(columns=['Utilities'])

# ── Group E: MSSubClass — cast to string ────────────────────────────────────
# Values 20, 30, 60, … are building-type labels; treating them as numbers
# implies a false ordering (e.g., type 60 is not "3× better" than type 20).
df['MSSubClass'] = df['MSSubClass'].astype(str)

# ── Group F: single-row true NaN → mode ────────────────────────────────────
for col in ['MSZoning', 'Electrical', 'KitchenQual',
            'Exterior1st', 'Exterior2nd', 'SaleType']:
    df[col] = df[col].fillna(df[col].mode()[0])

# ── Verify ───────────────────────────────────────────────────────────────────
remaining = df.drop(columns='SalePrice').isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print("✓ No NaN remaining in feature columns.")
else:
    print("⚠ Still missing:")
    print(remaining)

In [ ]:
# Before vs After comparison
before = data.isnull().sum()
after  = df.drop(columns='SalePrice').isnull().sum()

compare = pd.DataFrame({'Before': before, 'After': after}).dropna(how='all')
compare = compare[(compare['Before'] > 0) | (compare.get('After', pd.Series(0, index=compare.index)) > 0)]
display(compare.sort_values('Before', ascending=False))

---
## Step 5 — Define Features and Target

In [ ]:
y     = df['SalePrice'].copy()
X     = df.drop(columns='SalePrice').copy()
y_log = np.log1p(y)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numeric features  : {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Total             : {len(num_cols) + len(cat_cols)}")
print(f"\nSalePrice skewness (raw)     : {y.skew():.4f}")
print(f"SalePrice skewness (log1p)   : {y_log.skew():.4f}")

---
## Step 6 — Train / Test Split

We hold out 20 % as a completely unseen **test set**.  
All model choices and hyper-parameter tuning use cross-validation on the  
**training set only**, so the test set gives an honest generalisation estimate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=716
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

---
## Step 7 — Preprocessing Pipeline

| Column type | Imputer (safety net) | Transformer |
|---|---|---|
| Numeric | Median | `StandardScaler` |
| Categorical | Most-frequent | `OneHotEncoder` (ignore unknown categories) |

The `SimpleImputer` inside the pipeline is a **safety net** only — after our manual  
cleaning in Step 4, no NaN should remain. Wrapping the preprocessor in a `Pipeline`  
ensures no information leaks between the training and validation folds.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

print("Preprocessing pipeline configured.")

---
## Step 8 — Initial Model Comparison (5-fold CV)

Six model families are evaluated with default / reasonable starting hyper-parameters.  
Metric: RMSE on log-price — **lower is better**.

| Model | Regularisation | Key idea |
|---|---|---|
| `LinearRegression` | None | OLS baseline |
| `Ridge` | L2 | Shrinks all coefficients equally |
| `Lasso` | L1 | Sets irrelevant coefficients to zero |
| `ElasticNet` | L1 + L2 | Combines both — useful with many correlated features |
| `RandomForest` | — | Ensemble of trees, captures non-linearity |
| `XGBoost` | — | Gradient-boosted trees, often best on tabular data |

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=716)

models_init = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=10.0),
    'Lasso':            Lasso(alpha=0.001, max_iter=10000),
    'ElasticNet':       ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000),
    'RandomForest':     RandomForestRegressor(n_estimators=300, min_samples_leaf=2,
                                              random_state=716, n_jobs=-1),
    'XGBoost':          xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                          max_depth=4, subsample=0.8,
                                          colsample_bytree=0.8,
                                          random_state=716, n_jobs=-1, verbosity=0),
}

init_results = []
for name, model in models_init.items():
    pipe   = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')
    init_results.append({'Model': name, 'CV RMSE': -scores.mean(), 'Std': scores.std()})

init_df = pd.DataFrame(init_results).sort_values('CV RMSE')
print(init_df.to_string(index=False))

plt.figure(figsize=(9, 5))
plt.bar(init_df['Model'], init_df['CV RMSE'], yerr=init_df['Std'], capsize=4)
plt.ylabel("CV RMSE (log-price)")
plt.title("Initial Model Comparison — 5-fold CV (lower is better)")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## Step 9 — Hyper-parameter Tuning (Lab 7 style)

### 9a — Ridge: sweeping `alpha`

`alpha` controls the L2 penalty strength.  
Larger `alpha` → more shrinkage → simpler model.  
Too small → behaves like OLS (overfits); too large → underfits.

In [ ]:
ridge_alphas  = [0.1, 1, 5, 10, 50, 100]
ridge_cv_rmse = []

for alpha in ridge_alphas:
    pipe   = Pipeline([('preprocessor', preprocessor), ('model', Ridge(alpha=alpha))])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')
    ridge_cv_rmse.append(-scores.mean())
    print(f"Ridge  alpha={alpha:6.1f}  →  CV RMSE = {-scores.mean():.5f}")

best_ridge_alpha = ridge_alphas[int(np.argmin(ridge_cv_rmse))]
print(f"\n>>> Best Ridge alpha = {best_ridge_alpha}")

### 9b — Lasso: sweeping `alpha`

The L1 penalty drives some coefficients to exactly zero (automatic feature selection).  
Small `alpha` values are usually best when many features carry weak signal.

In [ ]:
lasso_alphas  = [0.0001, 0.0005, 0.001, 0.005, 0.01]
lasso_cv_rmse = []

for alpha in lasso_alphas:
    pipe   = Pipeline([('preprocessor', preprocessor),
                       ('model', Lasso(alpha=alpha, max_iter=10000))])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')
    lasso_cv_rmse.append(-scores.mean())
    print(f"Lasso  alpha={alpha:.4f}  →  CV RMSE = {-scores.mean():.5f}")

best_lasso_alpha = lasso_alphas[int(np.argmin(lasso_cv_rmse))]
print(f"\n>>> Best Lasso alpha = {best_lasso_alpha}")

### 9c — ElasticNet: sweeping `alpha` and `l1_ratio`

`l1_ratio = 1` → pure Lasso; `l1_ratio = 0` → pure Ridge.  
ElasticNet is particularly useful here because many dummy-encoded columns are  
correlated (e.g., neighbourhood dummies) — Lasso arbitrarily picks one, while  
ElasticNet can shrink the group together.

In [ ]:
en_grid    = [(0.001, 0.3), (0.001, 0.5), (0.001, 0.7),
              (0.0005, 0.3), (0.0005, 0.5)]
en_cv_rmse = []

for alpha, l1 in en_grid:
    pipe   = Pipeline([('preprocessor', preprocessor),
                       ('model', ElasticNet(alpha=alpha, l1_ratio=l1, max_iter=10000))])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')
    en_cv_rmse.append(-scores.mean())
    print(f"ElasticNet  alpha={alpha}, l1_ratio={l1}  →  CV RMSE = {-scores.mean():.5f}")

best_en = en_grid[int(np.argmin(en_cv_rmse))]
print(f"\n>>> Best ElasticNet: alpha={best_en[0]}, l1_ratio={best_en[1]}")

---
## Step 10 — Final CV Comparison with Tuned Hyper-parameters

In [ ]:
tuned_models = {
    'LinearRegression':                           LinearRegression(),
    f'Ridge(α={best_ridge_alpha})':               Ridge(alpha=best_ridge_alpha),
    f'Lasso(α={best_lasso_alpha})':               Lasso(alpha=best_lasso_alpha, max_iter=10000),
    f'ElasticNet(α={best_en[0]},l1={best_en[1]})': ElasticNet(alpha=best_en[0], l1_ratio=best_en[1],
                                                                max_iter=10000),
    'RandomForest':                               RandomForestRegressor(n_estimators=300,
                                                                        min_samples_leaf=2,
                                                                        random_state=716, n_jobs=-1),
    'XGBoost':                                    xgb.XGBRegressor(n_estimators=500,
                                                                     learning_rate=0.05,
                                                                     max_depth=4, subsample=0.8,
                                                                     colsample_bytree=0.8,
                                                                     random_state=716, n_jobs=-1,
                                                                     verbosity=0),
}

tuned_results = []
for name, model in tuned_models.items():
    pipe   = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                             scoring='neg_root_mean_squared_error')
    tuned_results.append({'Model': name, 'CV RMSE': -scores.mean(), 'Std': scores.std()})

tuned_df = pd.DataFrame(tuned_results).sort_values('CV RMSE')
print(tuned_df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.bar(tuned_df['Model'], tuned_df['CV RMSE'], yerr=tuned_df['Std'], capsize=4)
plt.ylabel("CV RMSE (log-price)")
plt.title("Tuned Model Comparison — 5-fold CV")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

---
## Step 11 — Test Set Evaluation: RMSE, MAE, R²

All models are retrained on the full training set and evaluated **once** on the  
held-out test set. The test set has not been touched at any earlier stage.

| Metric | Formula | Interpretation |
|---|---|---|
| **RMSE** | √mean((ŷ−y)²) | Penalises large errors more than MAE |
| **MAE** | mean(|ŷ−y|) | Average absolute error |
| **R²** | 1 − SS_res/SS_tot | Proportion of variance explained; 1.0 = perfect |

*All metrics are on the **log-price** scale.*

In [ ]:
test_results  = []
fitted_models = {}

for name, model in tuned_models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    test_results.append({'Model': name, 'Test RMSE': rmse, 'Test MAE': mae, 'Test R²': r2})
    fitted_models[name] = (pipe, y_pred)

test_df = pd.DataFrame(test_results).sort_values('Test RMSE')
print(test_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = [('Test RMSE', 'RMSE — lower is better'),
           ('Test MAE',  'MAE  — lower is better'),
           ('Test R²',   'R²   — higher is better')]

for ax, (col, title) in zip(axes, metrics):
    ax.bar(test_df['Model'], test_df[col])
    ax.set_title(title)
    ax.set_xticklabels(test_df['Model'], rotation=25, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
best_two = test_df.head(2)['Model'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, best_two):
    _, y_pred = fitted_models[name]
    ax.scatter(y_test, y_pred, alpha=0.4, s=12)
    lo, hi = float(y_test.min()), float(y_test.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
    ax.set_xlabel("Actual log(SalePrice)")
    ax.set_ylabel("Predicted log(SalePrice)")
    ax.set_title(name)
    ax.legend()

plt.suptitle("Actual vs Predicted — Two Best Models", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name in zip(axes, best_two):
    _, y_pred = fitted_models[name]
    residuals = y_test.values - y_pred
    ax.hist(residuals, bins=40, edgecolor='black')
    ax.axvline(0, color='red', linestyle='--', lw=1.5)
    ax.set_xlabel("Residual  (actual − predicted)")
    ax.set_title(f"Residuals — {name}")

plt.tight_layout()
plt.show()

---
## Step 12 — Interpretation of Findings

#### Data cleaning

The most consequential correction was recognising that columns like `PoolQC`, `FireplaceQu`, `GarageQual`, and the five basement quality columns use `NaN` to encode *absence of the feature*, not a recording gap. In the original notebook these five columns were simply dropped, discarding signal from 93–99 % of rows that genuinely had no pool, alley, etc. Recoding them as `'None'` (a new category) preserves that information: the model can now learn, for example, that having *no garage* is different from having an `Unf`-finished garage.

`GarageYrBlt`, `MasVnrArea`, `BsmtFullBath`, and `BsmtHalfBath` receive the same treatment on the numeric side — a house with no garage has a garage year of 0, not an unknown year.

`LotFrontage` (up to 55 % missing in some neighbourhoods) is genuinely unknown rather than absent, so neighbourhood-median imputation is used. `Utilities` is dropped because 99.9 % of houses share the same value — it adds noise but no signal. `MSSubClass` is recast to string because values like 20, 60, and 120 are building-type codes, not ordered numbers.

#### Model performance

After corrected preprocessing the three tuned regularised linear models (Ridge α=50, Lasso α=0.0005, ElasticNet α=0.001 l1=0.3) achieve test R² ≈ 0.93, matching or exceeding XGBoost. The L1 penalty in Lasso and ElasticNet zeroes out coefficients for the many low-information one-hot columns; Ridge retains all with shrunken weights; ElasticNet combines both strategies and handles correlated neighbourhood dummies most gracefully. Residuals are approximately symmetric around zero, confirming the log-transform was appropriate. RandomForest underperforms here because its hyperparameters were not tuned to match the linear models' tuning effort.